# Aviation MRO Local AI Assistant (Offline RAG Pipeline)
Overview

- This project is a secure, 100% offline Retrieval-Augmented Generation (RAG) prototype designed for Aviation Maintenance, Repair, and Overhaul (MRO) operations. It ingests technical documents—specifically the FAA Aviation Maintenance Technician (AMT) Handbook—and allows mechanics to query complex procedural data instantly using a locally hosted Large Language Model.
- By running entirely offline, this architecture ensures zero data leakage of proprietary or sensitive maintenance procedures, adhering to strict aerospace data security compliance.

Tech Stack
* **Language:** Python 3.14
* **LLM Engine:** Ollama (`phi3` model)
* **Embeddings:** Hugging Face (`all-MiniLM-L6-v2`)
* **Vector Database:** ChromaDB
* **Document Processing:** LangChain (`PyPDFLoader`, `RecursiveCharacterTextSplitter`)
* **Environment:** VS Code (Jupyter Notebook interface)



In [ ]:
!pip install langchain langchain-community pypdf chromadb sentence-transformers
!pip install langchain
import os
print("Python is currently looking in:", os.getcwd())
!pip install langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\egala\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Python is currently looking in: c:\Users\egala\Downloads\New_project



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\egala\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\egala\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## Document Ingestion and Semantic Chunking
- An LLM cannot process an entire 800-page manual at once. The PDF was loaded and split into overlapping chunks to preserve the context of the maintenance procedures.

In [ ]:
# 1. Load the document
loader = PyPDFLoader(r"C:\Users\egala\Downloads\New_project\manual.pdf.pdf")
pages = loader.load()

In [ ]:
# 2. Split the document
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

print(f"Split the manual into {len(chunks)} chunks!")

Split the manual into 136 chunks!


## Vector Database Creation
- To make the text chunks searchable by the AI, they were converted into mathematical embeddings and stored in a local Chroma vector database.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 3. Create the Embedding Model (Downloads a small, fast model)
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Build and Save the Vector Database locally
print("Building the task knowledge database...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./mro_chroma_db" # This saves the DB to your project folder!
)

print("Vector database built successfully! Ready for queries.")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4868.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building the task knowledge database...
Vector database built successfully! Ready for queries.


## Custom Retrieval & Generation (RAG) Loop
- To ensure maximum stability and avoid framework dependency errors, the standard LangChain QA wrappers were bypassed. A custom retrieval loop was engineered to query the Chroma database and dynamically inject the results directly into the local LLM's prompt.

In [ ]:
from langchain_community.llms import Ollama

# 1. Connect to your local Ollama model
print("Waking up the AI mechanic...")
llm = Ollama(model="phi3")

# 2. Define the question
query = "What are the common causes of corrosion on an aircraft?"
print(f"Question: {query}\n")
print("Searching the manual...")

# 3. THE PRO BYPASS: Manually search your Chroma database for the 3 best chunks
retrieved_docs = vector_db.similarity_search(query, k=3)

# Combine those 3 chunks into one big block of text
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# 4. Create a strict prompt so the AI only uses the manual
prompt = f"""You are an expert aviation mechanic. Use ONLY the following manual excerpts to answer the question.

Manual Excerpts:
{context}

Question: {query}
Answer:"""

print("Thinking...")

# 5. Get the final answer
response = llm.invoke(prompt)

print("\n--- AI Assistant Answer ---")
print(response)

Waking up the AI mechanic...
Question: What are the common causes of corrosion on an aircraft?

Searching the manual...
Thinking...

--- AI Assistant Answer ---
Common causes of corrosion on an aircraft include exposure to moisture, chemicals in the environment such as de-icing agents and road salts, humidity, temperature variations that can lead to condensation or freeze/thaw cycles, and even galvanic action due to different metals being electrically connected. All these factors contribute to creating a corrosive environment for metal structures of aircrafts when they are in use.
